# Stage 3: Structural Analysis (Extraction)

| Version | Date | Author | Description |
| --- | --- | --- | --- |
| 2.0.0 | 2026-01-26 | That Le | Complete Stage 3 documentation |

---

## Overview

Stage 3 is the **most complex stage** - it extracts all structural information from chart images.

### Input/Output

| Property | Value |
| --- | --- |
| **Input** | `Stage2Output` (List of `DetectedChart`) |
| **Output** | `Stage3Output` (List of `RawMetadata`) |
| **Key Components** | Classifier, OCR, Element Detector, Geometric Mapper |

### Processing Flow (Parallel)

```
Stage2Output (Cropped Charts)
       |
       v
+--------------------------------------------------+
|              PARALLEL PROCESSING                 |
|  +------------+  +------------+  +------------+  |
|  | CLASSIFIER |  |    OCR     |  |  ELEMENT   |  |
|  | ResNet-18  |  | PaddleOCR  |  | DETECTOR   |  |
|  | 94.66% acc |  | Multi-lang |  | K-Means +  |  |
|  +------------+  +------------+  | Contours   |  |
|        |              |          +------------+  |
|        v              v                |         |
|   ChartType       OCRText[]        Elements[]    |
+--------------------------------------------------+
       |
       v
+----------------------+
| GEOMETRIC ANALYSIS   |
| - Axis detection     |
| - Scale calibration  |
| - Coordinate mapping |
+----------------------+
       |
       v
RawMetadata (combined)
```

---

## Output Schema

```python
class RawMetadata(BaseModel):
    """Raw extracted data from Stage 3."""
    chart_id: str                        # From Stage 2
    chart_type: ChartType                # bar, line, pie, scatter...
    texts: List[OCRText]                 # All extracted text
    elements: List[ChartElement]         # Bars, points, slices
    axis_info: Optional[AxisInfo]        # Axis calibration data

class OCRText(BaseModel):
    """Extracted text element."""
    text: str                            # Text content
    bbox: BoundingBox                    # Location
    confidence: float                    # OCR confidence [0-1]
    role: Optional[str]                  # title/xlabel/ylabel/legend/value

class ChartElement(BaseModel):
    """Detected chart element."""
    element_type: str                    # bar, point, slice, line
    bbox: BoundingBox                    # Bounding box
    center: Point                        # Center point
    color: Optional[Color]               # Dominant color
    area_pixels: Optional[int]           # Area in pixels
```

---

## Configuration

In [ ]:
# ============================================================================
# EXECUTION CONTROL FLAG
# ============================================================================
# Set to True to execute actual examples
# WARNING: This loads heavy models (ResNet-18 ~43MB, PaddleOCR ~100MB)
# ============================================================================

EXECUTE_EXAMPLES = True  # <-- Change to True to run examples

print(f"Execution mode: {'ACTIVE' if EXECUTE_EXAMPLES else 'DOCUMENTATION ONLY'}")

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP (Always runs)
# ============================================================================
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

# Check for model files
model_files = [
    ("ResNet-18 Classifier", PROJECT_ROOT / "models" / "weights" / "resnet18_chart_classifier.onnx"),
    ("ResNet-18 PyTorch", PROJECT_ROOT / "models" / "weights" / "resnet18_chart_classifier.pth"),
]
print("\nModel files:")
for name, path in model_files:
    status = "FOUND" if path.exists() else "NOT FOUND"
    size = f"({path.stat().st_size / 1024 / 1024:.1f} MB)" if path.exists() else ""
    print(f"  [{status}] {name} {size}")

---

## 1. Chart Classification (ResNet-18)

**Model**: ResNet-18 trained on 8 chart types

### Performance Metrics

| Metric | Value |
| --- | --- |
| Training Accuracy | 94.66% |
| Integration Test | 93.75% (15/16 samples) |
| Inference Speed (ONNX) | 6.90ms mean (CPU) |
| Model Size | 42.64 MB |

### Supported Chart Types

| Index | Type | Description |
| --- | --- | --- |
| 0 | area | Area charts |
| 1 | bar | Vertical/horizontal bars |
| 2 | box | Box plots |
| 3 | heatmap | Heatmaps |
| 4 | histogram | Histograms |
| 5 | line | Line charts |
| 6 | pie | Pie/donut charts |
| 7 | scatter | Scatter plots |

In [ ]:
# ============================================================================
# DEMO: Chart Classification
# ============================================================================
if EXECUTE_EXAMPLES:
    import cv2
    import matplotlib.pyplot as plt
    
    from src.core_engine.stages.s3_extraction.classifier import (
        ResNet18Classifier, ClassifierConfig
    )
    
    # Initialize classifier
    config = ClassifierConfig(
        model_path=str(PROJECT_ROOT / "models" / "weights" / "resnet18_chart_classifier.onnx"),
        use_onnx=True,
    )
    classifier = ResNet18Classifier(config)
    
    print(f"Classifier loaded: {classifier.class_names}")
    
    # Find test images
    sample_dir = PROJECT_ROOT / "data" / "academic_dataset" / "images"
    if sample_dir.exists():
        sample_images = list(sample_dir.glob("*.png"))[:3]
        
        print("\nClassification Results:")
        print("=" * 50)
        
        for img_path in sample_images:
            img = cv2.imread(str(img_path))
            chart_type, confidence = classifier.classify(img)
            print(f"  {img_path.name}: {chart_type.value} ({confidence:.2%})")
else:
    print("[SKIPPED] Set EXECUTE_EXAMPLES = True to run classification")
    print("\nExpected output:")
    print("  Classifier loaded: ['area', 'bar', 'box', 'heatmap', 'histogram', 'line', 'pie', 'scatter']")
    print("  arxiv_chart_001.png: line (95.23%)")
    print("  arxiv_chart_002.png: bar (87.45%)")

---

## 2. OCR Text Extraction (PaddleOCR)

**Library**: PaddleOCR - Best accuracy for mixed languages

### Text Role Classification

Based on spatial position, extracted text is classified:

```
+------------------------------------------+
|               TITLE (top)                |
+------------------------------------------+
|        |                    | LEGEND     |
| Y-AXIS |                    | (top-right)|
| LABELS |    CHART AREA      |            |
|        |                    |            |
|        |   [VALUE LABELS]   |            |
+--------+--------------------+------------+
|            X-AXIS LABELS (bottom)        |
+------------------------------------------+
```

### Role Detection Rules

| Role | Position Rule |
| --- | --- |
| `title` | Top 15% of image, centered |
| `xlabel` | Bottom 15% of image |
| `ylabel` | Left 15% of image |
| `legend` | Right side or top-right corner |
| `value` | Near chart elements |

In [ ]:
# ============================================================================
# DEMO: OCR Text Extraction
# ============================================================================
if EXECUTE_EXAMPLES:
    from src.core_engine.stages.s3_extraction.ocr_engine import OCREngine, OCRConfig
    
    # Initialize OCR
    config = OCRConfig(
        languages=["en"],
        detect_orientation=False,
    )
    ocr_engine = OCREngine(config)
    
    print("OCR Engine initialized")
    
    # Test on sample
    if sample_dir.exists():
        sample_images = list(sample_dir.glob("*.png"))[:1]
        
        for img_path in sample_images:
            img = cv2.imread(str(img_path))
            result = ocr_engine.extract_text(img, chart_id=img_path.stem)
            
            print(f"\n{img_path.name}:")
            print(f"  Total texts: {len(result.texts)}")
            print("  By role:")
            
            # Group by role
            by_role = {}
            for t in result.texts:
                role = t.role or "unknown"
                if role not in by_role:
                    by_role[role] = []
                by_role[role].append(t.text)
            
            for role, texts in by_role.items():
                print(f"    [{role}]: {texts[:3]}{'...' if len(texts) > 3 else ''}")
else:
    print("[SKIPPED] Set EXECUTE_EXAMPLES = True to run OCR")
    print("\nExpected output:")
    print("  Total texts: 15")
    print("  By role:")
    print("    [title]: ['Quarterly Sales 2025']")
    print("    [xlabel]: ['Q1', 'Q2', 'Q3', 'Q4']")
    print("    [ylabel]: ['0', '100', '200', '300']")

---

## 3. Element Detection (K-Means + Contours)

### Detection Approach

| Chart Type | Detection Method |
| --- | --- |
| Bar | Contour analysis + aspect ratio |
| Line | Skeleton extraction + keypoints |
| Pie | Ellipse detection + angle analysis |
| Scatter | Small circular contours |

### K-Means Color Segmentation

For charts with multiple colors (stacked bars, grouped bars):

```python
# 1. Extract dominant colors using K-Means
colors = kmeans(image, n_clusters=5)

# 2. Create mask for each color
for color in colors:
    mask = color_mask(image, color, tolerance=30)
    contours = find_contours(mask)
    elements.extend(contours)
```

In [ ]:
# ============================================================================
# DEMO: Element Detection
# ============================================================================
if EXECUTE_EXAMPLES:
    from src.core_engine.stages.s3_extraction.element_detector import (
        ElementDetector, ElementDetectorConfig
    )
    
    # Initialize detector
    config = ElementDetectorConfig(
        detect_bars=True,
        detect_markers=True,
        use_color_segmentation=True,  # K-Means for better detection
        color_saturation_threshold=30,
        min_bar_area=100,
    )
    detector = ElementDetector(config)
    
    print("Element Detector initialized")
    print(f"  Color segmentation: {config.use_color_segmentation}")
    
    # Test on sample
    if sample_dir.exists():
        sample_images = list(sample_dir.glob("*.png"))[:1]
        
        for img_path in sample_images:
            img = cv2.imread(str(img_path))
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
            
            result = detector.detect(binary, img, chart_id=img_path.stem)
            
            print(f"\n{img_path.name}:")
            print(f"  Bars: {len(result.bars)}")
            print(f"  Markers: {len(result.markers)}")
            print(f"  Lines: {len(result.lines)}")
else:
    print("[SKIPPED] Set EXECUTE_EXAMPLES = True to run element detection")
    print("\nExpected output:")
    print("  arxiv_chart_001.png:")
    print("    Bars: 4")
    print("    Markers: 0")
    print("    Lines: 0")

---

## 4. Preprocessing Pipeline

Before element detection, images go through preprocessing:

### Steps

1. **Negative Transform**: Invert colors (better for skeleton extraction)
2. **Adaptive Threshold**: Handle varying lighting
3. **Morphological Operations**: Clean up noise
4. **Text Masking**: Remove text regions before element detection

In [ ]:
# ============================================================================
# DEMO: Preprocessing Visualization
# ============================================================================
if EXECUTE_EXAMPLES:
    import numpy as np
    
    if sample_dir.exists():
        img_path = list(sample_dir.glob("*.png"))[0]
        img = cv2.imread(str(img_path))
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        # Preprocessing steps
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        negative = 255 - gray
        _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
        edges = cv2.Canny(gray, 50, 150)
        
        # Visualize
        fig, axes = plt.subplots(2, 3, figsize=(15, 10))
        
        axes[0, 0].imshow(img_rgb)
        axes[0, 0].set_title("1. Original")
        axes[0, 0].axis('off')
        
        axes[0, 1].imshow(gray, cmap='gray')
        axes[0, 1].set_title("2. Grayscale")
        axes[0, 1].axis('off')
        
        axes[0, 2].imshow(negative, cmap='gray')
        axes[0, 2].set_title("3. Negative")
        axes[0, 2].axis('off')
        
        axes[1, 0].imshow(binary, cmap='gray')
        axes[1, 0].set_title("4. Binary (Otsu)")
        axes[1, 0].axis('off')
        
        axes[1, 1].imshow(edges, cmap='gray')
        axes[1, 1].set_title("5. Edges (Canny)")
        axes[1, 1].axis('off')
        
        # Morphological cleaning
        kernel = np.ones((3, 3), np.uint8)
        cleaned = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)
        axes[1, 2].imshow(cleaned, cmap='gray')
        axes[1, 2].set_title("6. Morphological Clean")
        axes[1, 2].axis('off')
        
        plt.suptitle("Preprocessing Pipeline Steps", fontsize=14)
        plt.tight_layout()
        plt.show()
else:
    print("[SKIPPED] Set EXECUTE_EXAMPLES = True to see preprocessing")
    print("\nPreprocessing steps:")
    print("  1. Original -> 2. Grayscale -> 3. Negative")
    print("  4. Binary (Otsu) -> 5. Edges (Canny) -> 6. Morphological Clean")

---

## 5. Complete Stage 3 Processing

### Usage Example

```python
from src.core_engine.stages.s3_extraction import Stage3Extraction, ExtractionConfig

# Initialize with full configuration
config = ExtractionConfig(
    use_ml_classifier=True,        # ResNet-18 (94.66% accuracy)
    enable_ocr=True,               # PaddleOCR
    enable_element_detection=True, # K-Means + Contours
    use_color_segmentation=True,   # Better for stacked bars
    enable_vectorization=False,    # Optional: RDP vectorization
)
stage3 = Stage3Extraction(config)

# Process single image directly
result = stage3.process_image(img_bgr, chart_id="chart_001")

# Or process Stage2Output
stage3_output = stage3.process(stage2_output)
```

In [ ]:
# ============================================================================
# DEMO: Full Stage 3 Processing
# ============================================================================
if EXECUTE_EXAMPLES:
    from src.core_engine.stages.s3_extraction import Stage3Extraction, ExtractionConfig
    
    # Initialize Stage 3
    config = ExtractionConfig(
        use_ml_classifier=True,
        enable_ocr=True,
        enable_element_detection=True,
        use_color_segmentation=True,
    )
    stage3 = Stage3Extraction(config=config)
    
    print("Stage 3 initialized")
    print(f"  Classifier: {'ResNet-18' if config.use_ml_classifier else 'Simple'}")
    print(f"  OCR: {config.ocr_engine}")
    print(f"  Color Segmentation: {config.use_color_segmentation}")
    
    # Process sample image
    if sample_dir.exists():
        img_path = list(sample_dir.glob("*.png"))[0]
        img = cv2.imread(str(img_path))
        
        result = stage3.process_image(img, chart_id=img_path.stem)
        
        print("\n" + "=" * 50)
        print("Stage 3 Results")
        print("=" * 50)
        print(f"Chart ID: {result.chart_id}")
        print(f"Chart Type: {result.chart_type.value}")
        print(f"Texts: {len(result.texts)}")
        print(f"Elements: {len(result.elements)}")
        
        # Show title if detected
        titles = [t for t in result.texts if t.role == "title"]
        if titles:
            print(f"Title: {titles[0].text}")
else:
    print("[SKIPPED] Set EXECUTE_EXAMPLES = True to run full Stage 3")
    print("\nExpected output:")
    print("  Chart Type: line")
    print("  Texts: 12")
    print("  Elements: 5")
    print("  Title: 'Performance Comparison'")

---

## 6. Visualization: Detection Results

In [ ]:
# ============================================================================
# DEMO: Visualize Stage 3 Results
# ============================================================================
if EXECUTE_EXAMPLES:
    # Create visualization
    img_viz = img.copy()
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Draw elements
    for elem in result.elements:
        x1, y1 = int(elem.bbox.x_min), int(elem.bbox.y_min)
        x2, y2 = int(elem.bbox.x_max), int(elem.bbox.y_max)
        cv2.rectangle(img_viz, (x1, y1), (x2, y2), (0, 255, 255), 2)
        cv2.circle(img_viz, (int(elem.center.x), int(elem.center.y)), 4, (0, 0, 255), -1)
    
    # Draw text boxes
    role_colors = {
        'title': (255, 0, 0),
        'xlabel': (0, 255, 0),
        'ylabel': (0, 0, 255),
        'legend': (255, 165, 0),
        'value': (128, 0, 128),
    }
    for text in result.texts[:10]:  # Limit for clarity
        color = role_colors.get(text.role, (128, 128, 128))
        x1, y1 = int(text.bbox.x_min), int(text.bbox.y_min)
        x2, y2 = int(text.bbox.x_max), int(text.bbox.y_max)
        cv2.rectangle(img_viz, (x1, y1), (x2, y2), color, 1)
    
    # Display
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    
    axes[0].imshow(img_rgb)
    axes[0].set_title("Original")
    axes[0].axis('off')
    
    axes[1].imshow(cv2.cvtColor(img_viz, cv2.COLOR_BGR2RGB))
    axes[1].set_title(f"Stage 3 Results\nType: {result.chart_type.value}, Elements: {len(result.elements)}")
    axes[1].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    # Legend
    print("\nColor Legend:")
    print("  Cyan boxes = Elements (bars/points)")
    print("  Red = Title, Green = X-labels, Blue = Y-labels")
    print("  Orange = Legend, Purple = Values")
else:
    print("[SKIPPED] Set EXECUTE_EXAMPLES = True to visualize results")

---

## Summary

### Stage 3 Components

| Component | Technology | Accuracy/Speed |
| --- | --- | --- |
| Classifier | ResNet-18 (ONNX) | 94.66%, 6.9ms |
| OCR | PaddleOCR | High accuracy, multi-lang |
| Element Detector | K-Means + Contours | Improved for stacked bars |
| Preprocessor | OpenCV | Negative + CLAHE |

### Configuration Options

| Parameter | Default | Description |
| --- | --- | --- |
| `use_ml_classifier` | True | Use ResNet-18 (vs SimpleClassifier) |
| `enable_ocr` | True | Run OCR text extraction |
| `ocr_engine` | paddleocr | OCR backend |
| `enable_element_detection` | True | Detect chart elements |
| `use_color_segmentation` | True | K-Means for colors |
| `enable_vectorization` | False | RDP line vectorization |

### Error Handling

| Error | Severity | Action |
| --- | --- | --- |
| Classifier model missing | WARNING | Use SimpleClassifier |
| OCR failure | WARNING | Return empty texts |
| No elements detected | NORMAL | Return empty list |
| Classification uncertain | WARNING | Default to "unknown" |

---

**Previous**: [Stage 2 - Detection](02_stage2_detection.ipynb)  
**Next**: [Stage 4 - Reasoning](04_stage4_reasoning.ipynb)